In [1]:
# (Timing: ~ 2200 s)

import tarfile, re, time

keggdir = '/home/feiran' # the folder of the downloaded KEGG database

# organism info
fhand = open(keggdir + '/kegg/genes/misc/taxonomy')

org2kingdom = dict()
org2species = dict()

for line in fhand:
    if line.startswith('# Eukaryotes'):
        kingdom = 'E'
    elif line.startswith('## Bacteria'):
        kingdom = 'B'
    elif line.startswith('## Archaea'):
        kingdom = 'A'
    elif not line.startswith('#'):
        org2kingdom[line.split('\t')[1]] = kingdom
        org2species[line.split('\t')[1]] = line.split('\t')[3].rstrip()
fhand.close()

tar = tarfile.open(keggdir + '/kegg/ligand/reaction.tar.gz', 'r:gz')

# reaction-compound relationship
f_rc = tar.extractfile('reaction/reaction.lst')
rxn2cmpd = dict() # reaction to compound
# metabolites to be removed
rmMets1 = ['C00001','C00007','C00012','C00017','C00039','C00046','C00080']
rmMets2 = ['C00002','C00003','C00004','C00005','C00006','C00008','C00009',
           'C00010','C00013','C00016','C00019','C00020','C00021','C00035',
           'C00044','C00061','C00144','C00390','C00399','C01352','C01847']
for line in f_rc:
    r_id = line.decode().split(':')[0]
    cmpd_tmp = re.findall('[A-Z]\d{5}', line.decode().split(':')[1])
    cmpd_tmp1 = [ x for x in cmpd_tmp if x not in rmMets1]
    cmpd_tmp2 = [ x for x in cmpd_tmp1 if x not in rmMets2]
    if len(cmpd_tmp1) > 0:
        if len(cmpd_tmp2) > 0:
            cmpd = cmpd_tmp2
        else:
            cmpd = cmpd_tmp1
        rxn2cmpd[r_id] = cmpd
f_rc.close()

# reaction-EC relationship
f_rec = tar.extractfile('reaction/reaction')
rxn2ec = dict() # reaction to ec
flag = False # used for EC numbers in multiple lines
for line in f_rec:
    
    if line.decode().startswith('ENTRY'):
        r_id = re.findall('R\d{5}', line.decode())[0]
        rxn2ec[r_id] = []
    elif line.decode().startswith('ENZYME'):
        rxn2ec[r_id] += re.findall('\S+\.\S+\.\S+\.\S+', line.decode())
        if line.decode().endswith(' \n'):
            flag = True
    # for EC numbers in multiple lines
    if flag and line.decode().startswith(' '):
        rxn2ec[r_id] += re.findall('\S+\.\S+\.\S+\.\S+', line.decode())
        if line.decode().endswith(' \n'):
            flag = True
        else:
            flag = False

f_rec.close()
tar.close()


# KO- enzyme/reaction relationship
tar = tarfile.open(keggdir + '/kegg/genes/ko.tar.gz', 'r:gz')

# dict for KO-EC and KO-reaction
def generate_KOdict(file_name):
    KOdict = dict()
    f = tar.extractfile(file_name)
    for line in f:
        ko = line.decode().split('\t')[0][3:]
        ko_vals = line.decode().split('\t')[1].rstrip()[3:]
        if ko not in KOdict:
            KOdict[ko] = [ko_vals]
        else:
            KOdict[ko].append(ko_vals)
    f.close()
    return KOdict
ko2ec = generate_KOdict('ko/ko_enzyme.list') # KO to EC (not used in this project as there exists reaction to EC relationship)
ko2rxn = generate_KOdict('ko/ko_reaction.list') # KO to reaction


# write file for each organism based on KO-genes relationship

f_kg = tar.extractfile('ko/ko_genes.list')

org_list = []
time_start = time.time()

for line in f_kg:
    ko = line.decode().split('\t')[0][3:]
    if ko in ko2rxn: # only include genes with reactions
        gene_id = line.decode().split('\t')[1].rstrip()
        org_id = gene_id.split(':')[0]
        
        if org_id in org2kingdom:
            fout = open('output/' + org_id + '.txt', 'a')
            rxn_list = ko2rxn[ko]
            for rxn in rxn_list:
                ec_list = ';'.join(rxn2ec[rxn]) # (multiple ec numbers are grouped in one cell but could be splitted in future)
                cmpd_list = rxn2cmpd[rxn]
                for cmpd in cmpd_list:
                    line_to_write = gene_id+'\t'+org_id+'\t'+org2kingdom[org_id]+'\t'+rxn+'\t'+ec_list+'\t'+cmpd+'\n'
                    fout.write(line_to_write)

            fout.close()

            if org_id not in org_list:
                org_list += [org_id]
                if len(org_list)%100 == 0: 
                    time_end = time.time()
                    print('Done for', len(org_list), '/', len(org2kingdom), 'organisms', '[time cost:', round(time_end-time_start), 's]')

f_kg.close()
tar.close()

<>:37: SyntaxWarning: invalid escape sequence '\d'
<>:55: SyntaxWarning: invalid escape sequence '\d'
<>:58: SyntaxWarning: invalid escape sequence '\S'
<>:63: SyntaxWarning: invalid escape sequence '\S'
<>:37: SyntaxWarning: invalid escape sequence '\d'
<>:55: SyntaxWarning: invalid escape sequence '\d'
<>:58: SyntaxWarning: invalid escape sequence '\S'
<>:63: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1145966/2715527081.py:37: SyntaxWarning: invalid escape sequence '\d'
  cmpd_tmp = re.findall('[A-Z]\d{5}', line.decode().split(':')[1])
/tmp/ipykernel_1145966/2715527081.py:55: SyntaxWarning: invalid escape sequence '\d'
  r_id = re.findall('R\d{5}', line.decode())[0]
/tmp/ipykernel_1145966/2715527081.py:58: SyntaxWarning: invalid escape sequence '\S'
  rxn2ec[r_id] += re.findall('\S+\.\S+\.\S+\.\S+', line.decode())
/tmp/ipykernel_1145966/2715527081.py:63: SyntaxWarning: invalid escape sequence '\S'
  rxn2ec[r_id] += re.findall('\S+\.\S+\.\S+\.\S+', line.decode())


Done for 100 / 10769 organisms [time cost: 10 s]
Done for 200 / 10769 organisms [time cost: 15 s]
Done for 300 / 10769 organisms [time cost: 20 s]
Done for 400 / 10769 organisms [time cost: 26 s]
Done for 500 / 10769 organisms [time cost: 32 s]
Done for 600 / 10769 organisms [time cost: 36 s]
Done for 700 / 10769 organisms [time cost: 40 s]
Done for 800 / 10769 organisms [time cost: 49 s]
Done for 900 / 10769 organisms [time cost: 58 s]
Done for 1000 / 10769 organisms [time cost: 61 s]
Done for 1100 / 10769 organisms [time cost: 64 s]
Done for 1200 / 10769 organisms [time cost: 67 s]
Done for 1300 / 10769 organisms [time cost: 70 s]
Done for 1400 / 10769 organisms [time cost: 74 s]
Done for 1500 / 10769 organisms [time cost: 77 s]
Done for 1600 / 10769 organisms [time cost: 81 s]
Done for 1700 / 10769 organisms [time cost: 84 s]
Done for 1800 / 10769 organisms [time cost: 87 s]
Done for 1900 / 10769 organisms [time cost: 89 s]
Done for 2000 / 10769 organisms [time cost: 92 s]
Done for 

In [3]:
import re
output = '''  1%|█                                                                                                                                               | 80/10765 [00:06<11:00, 16.18it/s]Warning: No PEP file found for lsal, skipping.
  2%|███▏                                                                                                                                           | 237/10765 [00:17<15:46, 11.12it/s]Warning: No PEP file found for dtx, skipping.
  4%|█████▌                                                                                                                                         | 417/10765 [00:32<12:42, 13.57it/s]Warning: No PEP file found for hsz, skipping.
Warning: No PEP file found for hut, skipping.
  6%|█████████                                                                                                                                      | 679/10765 [00:50<11:59, 14.03it/s]Warning: No PEP file found for adl, skipping.
  6%|█████████▏                                                                                                                                     | 687/10765 [00:50<09:32, 17.61it/s]Warning: No PEP file found for saq, skipping.
  7%|█████████▉                                                                                                                                     | 744/10765 [01:00<33:43,  4.95it/s]Warning: No PEP file found for mdp, skipping.
 10%|██████████████▊                                                                                                                               | 1125/10765 [01:31<22:06,  7.26it/s]Warning: No PEP file found for bft, skipping.
 11%|███████████████▌                                                                                                                              | 1178/10765 [01:35<09:36, 16.62it/s]Warning: No PEP file found for hhg, skipping.
 11%|███████████████▋                                                                                                                              | 1191/10765 [01:35<09:08, 17.45it/s]Warning: No PEP file found for fml, skipping.
 12%|█████████████████▏                                                                                                                            | 1300/10765 [01:44<14:44, 10.70it/s]Warning: No PEP file found for amd, skipping.
 12%|█████████████████▏                                                                                                                            | 1307/10765 [01:45<15:51,  9.94it/s]Warning: No PEP file found for sabi, skipping.
 12%|█████████████████▋                                                                                                                            | 1341/10765 [01:48<17:03,  9.21it/s]Warning: No PEP file found for rgo, skipping.
 14%|███████████████████▉                                                                                                                          | 1510/10765 [02:00<16:10,  9.54it/s]Warning: No PEP file found for aflo, skipping.
 14%|████████████████████▎                                                                                                                         | 1542/10765 [02:01<07:29, 20.53it/s]Warning: No PEP file found for camz, skipping.
 15%|█████████████████████▌                                                                                                                        | 1631/10765 [02:09<07:51, 19.37it/s]Warning: No PEP file found for gva, skipping.
 15%|█████████████████████▋                                                                                                                        | 1641/10765 [02:10<11:07, 13.67it/s]Warning: No PEP file found for smir, skipping.
 16%|██████████████████████▌                                                                                                                       | 1706/10765 [02:16<11:00, 13.72it/s]Warning: No PEP file found for bchr, skipping.
 16%|██████████████████████▉                                                                                                                       | 1743/10765 [02:18<11:20, 13.25it/s]Warning: No PEP file found for bbro, skipping.
 16%|███████████████████████                                                                                                                       | 1747/10765 [02:19<14:33, 10.33it/s]Error: Directory /home/feiran/kegg/genes/organisms/mgy does not exist.
 19%|██████████████████████████▌                                                                                                                   | 2016/10765 [02:40<07:15, 20.08it/s]Warning: No PEP file found for amyy, skipping.
 19%|███████████████████████████▍                                                                                                                  | 2076/10765 [02:45<08:58, 16.15it/s]Warning: No PEP file found for lxy, skipping.
 20%|███████████████████████████▉                                                                                                                  | 2122/10765 [02:48<07:37, 18.89it/s]Warning: No PEP file found for psev, skipping.
 20%|████████████████████████████                                                                                                                  | 2127/10765 [02:48<05:36, 25.70it/s]Warning: No PEP file found for mmae, skipping.
 24%|██████████████████████████████████▏                                                                                                           | 2591/10765 [03:32<08:08, 16.74it/s]Warning: No PEP file found for bcd, skipping.
 26%|█████████████████████████████████████▎                                                                                                        | 2827/10765 [03:57<12:04, 10.96it/s]Warning: No PEP file found for lar, skipping.
 29%|████████████████████████████████████████▌                                                                                                     | 3077/10765 [04:14<07:06, 18.01it/s]Warning: No PEP file found for brea, skipping.
 29%|████████████████████████████████████████▋                                                                                                     | 3081/10765 [04:14<06:02, 21.17it/s]Warning: No PEP file found for ptd, skipping.
 29%|█████████████████████████████████████████                                                                                                     | 3115/10765 [04:16<06:06, 20.90it/s]Error: Directory /home/feiran/kegg/genes/organisms/bpk does not exist.
 31%|███████████████████████████████████████████▌                                                                                                  | 3298/10765 [04:37<17:35,  7.08it/s]Warning: No PEP file found for slit, skipping.
 31%|████████████████████████████████████████████▎                                                                                                 | 3357/10765 [04:42<14:14,  8.67it/s]Warning: No PEP file found for cnk, skipping.
 34%|████████████████████████████████████████████████▊                                                                                             | 3700/10765 [05:06<06:24, 18.39it/s]Error: Directory /home/feiran/kegg/genes/organisms/nul does not exist.
 36%|██████████████████████████████████████████████████▋                                                                                           | 3845/10765 [05:17<17:52,  6.45it/s]Warning: No PEP file found for asf, skipping.
 36%|██████████████████████████████████████████████████▊                                                                                           | 3855/10765 [05:18<08:35, 13.41it/s]Warning: No PEP file found for ctlm, skipping.
 37%|████████████████████████████████████████████████████                                                                                          | 3946/10765 [05:23<05:49, 19.54it/s]Warning: No PEP file found for rhof, skipping.
 40%|████████████████████████████████████████████████████████▎                                                                                     | 4268/10765 [05:50<04:58, 21.77it/s]Warning: No PEP file found for mmb, skipping.
 44%|██████████████████████████████████████████████████████████████▍                                                                               | 4731/10765 [06:28<06:25, 15.64it/s]Warning: No PEP file found for rhg, skipping.
 45%|███████████████████████████████████████████████████████████████▍                                                                              | 4807/10765 [06:34<06:12, 16.00it/s]Error: Directory /home/feiran/kegg/genes/organisms/pdul does not exist.
 46%|█████████████████████████████████████████████████████████████████▌                                                                            | 4975/10765 [06:46<03:53, 24.80it/s]Error: Directory /home/feiran/kegg/genes/organisms/rge does not exist.
 47%|███████████████████████████████████████████████████████████████████                                                                           | 5084/10765 [06:59<14:51,  6.37it/s]Warning: No PEP file found for ctrv, skipping.
 49%|████████████████████████████████████████████████████████████████████▉                                                                         | 5229/10765 [07:08<05:16, 17.51it/s]Warning: No PEP file found for leri, skipping.
 50%|██████████████████████████████████████████████████████████████████████▉                                                                       | 5378/10765 [07:20<06:08, 14.63it/s]Warning: No PEP file found for eja, skipping.
 53%|██████████████████████████████████████████████████████████████████████████▋                                                                   | 5658/10765 [07:42<06:45, 12.60it/s]Warning: No PEP file found for mbai, skipping.
 53%|██████████████████████████████████████████████████████████████████████████▊                                                                   | 5675/10765 [07:44<06:38, 12.78it/s]Warning: No PEP file found for ram, skipping.
 53%|██████████████████████████████████████████████████████████████████████████▉                                                                   | 5680/10765 [07:45<08:48,  9.62it/s]Warning: No PEP file found for cins, skipping.
 53%|███████████████████████████████████████████████████████████████████████████▌                                                                  | 5731/10765 [07:48<07:16, 11.54it/s]Warning: No PEP file found for cpl, skipping.
 54%|████████████████████████████████████████████████████████████████████████████▌                                                                 | 5806/10765 [07:53<03:37, 22.84it/s]Warning: No PEP file found for vaz, skipping.
 55%|██████████████████████████████████████████████████████████████████████████████                                                                | 5918/10765 [08:05<05:17, 15.28it/s]Warning: No PEP file found for rmc, skipping.
 55%|██████████████████████████████████████████████████████████████████████████████▏                                                               | 5932/10765 [08:06<07:44, 10.41it/s]Warning: No PEP file found for tpx, skipping.
 55%|██████████████████████████████████████████████████████████████████████████████▎                                                               | 5941/10765 [08:07<05:01, 15.98it/s]Warning: No PEP file found for agif, skipping.
 56%|███████████████████████████████████████████████████████████████████████████████▋                                                              | 6038/10765 [08:14<04:25, 17.78it/s]Warning: No PEP file found for nad, skipping.
 60%|█████████████████████████████████████████████████████████████████████████████████████▏                                                        | 6456/10765 [08:50<03:34, 20.09it/s]Warning: No PEP file found for cglo, skipping.
 61%|██████████████████████████████████████████████████████████████████████████████████████▌                                                       | 6560/10765 [08:56<03:42, 18.91it/s]Warning: No PEP file found for bamt, skipping.
 61%|███████████████████████████████████████████████████████████████████████████████████████                                                       | 6599/10765 [09:02<11:26,  6.07it/s]Warning: No PEP file found for tvd, skipping.
 63%|████████████████████████████████████████████████████████████████████████████████████████▉                                                     | 6744/10765 [09:17<06:57,  9.62it/s]Warning: No PEP file found for eryt, skipping.
 64%|██████████████████████████████████████████████████████████████████████████████████████████▌                                                   | 6867/10765 [09:26<05:06, 12.71it/s]Warning: No PEP file found for mcaz, skipping.
 64%|███████████████████████████████████████████████████████████████████████████████████████████▍                                                  | 6935/10765 [09:32<03:55, 16.28it/s]Warning: No PEP file found for hms, skipping.
 65%|████████████████████████████████████████████████████████████████████████████████████████████▎                                                 | 6996/10765 [09:37<04:21, 14.41it/s]Warning: No PEP file found for lfl, skipping.
 70%|███████████████████████████████████████████████████████████████████████████████████████████████████▍                                          | 7542/10765 [10:23<03:16, 16.37it/s]Warning: No PEP file found for cmic, skipping.
 70%|████████████████████████████████████████████████████████████████████████████████████████████████████                                          | 7588/10765 [10:27<07:18,  7.24it/s]Warning: No PEP file found for pphe, skipping.
 71%|████████████████████████████████████████████████████████████████████████████████████████████████████▏                                         | 7591/10765 [10:27<05:32,  9.54it/s]Warning: No PEP file found for dsc, skipping.
 71%|████████████████████████████████████████████████████████████████████████████████████████████████████▏                                         | 7597/10765 [10:28<08:08,  6.49it/s]Warning: No PEP file found for rlb, skipping.
 72%|██████████████████████████████████████████████████████████████████████████████████████████████████████▏                                       | 7748/10765 [10:45<02:49, 17.80it/s]Warning: No PEP file found for mnr, skipping.
 74%|█████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                    | 7985/10765 [11:05<03:01, 15.33it/s]Warning: No PEP file found for shs, skipping.
 77%|████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                                 | 8250/10765 [11:25<02:21, 17.83it/s]Warning: No PEP file found for msea, skipping.
 79%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                             | 8503/10765 [11:47<02:37, 14.40it/s]Warning: No PEP file found for xgl, skipping.
Warning: No PEP file found for lze, skipping.
 82%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                          | 8774/10765 [12:10<02:29, 13.31it/s]Warning: No PEP file found for haxz, skipping.
 82%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉                         | 8864/10765 [12:17<02:02, 15.47it/s]Warning: No PEP file found for bpyo, skipping.
 84%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                       | 9004/10765 [12:29<03:55,  7.48it/s]Warning: No PEP file found for psee, skipping.
 84%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍                      | 9050/10765 [12:32<02:28, 11.53it/s]Error: Directory /home/feiran/kegg/genes/organisms/fcy does not exist.
 86%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                    | 9214/10765 [12:46<02:15, 11.48it/s]Warning: No PEP file found for mico, skipping.
 93%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████          | 10004/10765 [13:55<01:00, 12.55it/s]Warning: No PEP file found for eds, skipping.
 93%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏         | 10014/10765 [13:56<01:28,  8.45it/s]Warning: No PEP file found for sng, skipping.
 95%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎      | 10259/10765 [14:21<01:26,  5.88it/s]Warning: No PEP file found for ccag, skipping.
 95%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌      | 10277/10765 [14:23<00:36, 13.42it/s]Warning: No PEP file found for ccoo, skipping.
 96%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏     | 10325/10765 [14:28<01:28,  4.99it/s]Error: Directory /home/feiran/kegg/genes/organisms/lch does not exist.
 97%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████     | 10392/10765 [14:35<01:03,  5.90it/s]Warning: No PEP file found for bal, skipping.
 97%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎   | 10487/10765 [14:43<00:17, 15.83it/s]Warning: No PEP file found for aof, skipping.
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 10753/10765 [15:07<00:00, 16.50it/s]Warning: No PEP file found for eae, skipping.
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10765/10765 [15:08<00:00, 11.85it/s]'''
warning_keys = re.findall(r'Warning: No PEP file found for (\w+), skipping', output)
error_keys = re.findall(r'Error: Directory ([\w/]+) does not exist', output)

# 打印提取的关键内容
print("\n=== Extracted Warning Keys ===")
for key in warning_keys:
    print(key)

print("\n=== Extracted Error Keys ===")
for key in error_keys:
    print(key)

# 将提取结果存入列表
warnings_list = warning_keys
errors_list = error_keys

# 打印最终的列表
print("\nWarnings List:", warnings_list)
print("Errors List:", errors_list)


=== Extracted Warning Keys ===
lsal
dtx
hsz
hut
adl
saq
mdp
bft
hhg
fml
amd
sabi
rgo
aflo
camz
gva
smir
bchr
bbro
amyy
lxy
psev
mmae
bcd
lar
brea
ptd
slit
cnk
asf
ctlm
rhof
mmb
rhg
ctrv
leri
eja
mbai
ram
cins
cpl
vaz
rmc
tpx
agif
nad
cglo
bamt
tvd
eryt
mcaz
hms
lfl
cmic
pphe
dsc
rlb
mnr
shs
msea
xgl
lze
haxz
bpyo
psee
mico
eds
sng
ccag
ccoo
bal
aof
eae

=== Extracted Error Keys ===
/home/feiran/kegg/genes/organisms/mgy
/home/feiran/kegg/genes/organisms/bpk
/home/feiran/kegg/genes/organisms/nul
/home/feiran/kegg/genes/organisms/pdul
/home/feiran/kegg/genes/organisms/rge
/home/feiran/kegg/genes/organisms/fcy
/home/feiran/kegg/genes/organisms/lch

Warnings List: ['lsal', 'dtx', 'hsz', 'hut', 'adl', 'saq', 'mdp', 'bft', 'hhg', 'fml', 'amd', 'sabi', 'rgo', 'aflo', 'camz', 'gva', 'smir', 'bchr', 'bbro', 'amyy', 'lxy', 'psev', 'mmae', 'bcd', 'lar', 'brea', 'ptd', 'slit', 'cnk', 'asf', 'ctlm', 'rhof', 'mmb', 'rhg', 'ctrv', 'leri', 'eja', 'mbai', 'ram', 'cins', 'cpl', 'vaz', 'rmc', 'tpx', 'agi

In [ ]:
['mgy', 'bpk', 'nul', 'pdul', 'rge', 'fcy', 'lch']

In [ ]:
import os

# FTP 下载的基本命令模板
ftp_base_url = "https://ftp.kegg.net/kegg/genes/organisms/"
wget_command_template = "wget --user=ac02489 --password=fovxaw-3Qijmi-rozzid -r --no-parent -nH -P {download_path} {url}"

# 下载路径
base_download_path = "/home/feiran/"

# 补充下载 warning_list 中的 .pep.gz 文件
for species in warnings_list:
    species_folder = f"{ftp_base_url}{species}/"
    download_path = os.path.join(base_download_path, "kegg/genes/organisms", species)
    
    # 确保目标文件夹存在
    os.makedirs(download_path, exist_ok=True)
    
    # 下载所有 .pep.gz 文件
    wget_command = wget_command_template.format(
        download_path=download_path,
        url=f"{species_folder}*.pep.gz"
    )
    print(f"Downloading missing .pep.gz files for species: {species}")
    os.system(wget_command)

# 补充下载 error_list 中的整个物种文件夹
for species in errors_list:
    species_folder = f"{ftp_base_url}{species}/"
    download_path = os.path.join(base_download_path, "kegg/genes/organisms")
    
    # 下载整个物种文件夹
    wget_command = wget_command_template.format(
        download_path=download_path,
        url=species_folder
    )
    print(f"Downloading entire species folder for: {species}")
    os.system(wget_command)

print("All downloads completed.")

--2025-04-21 15:29:54--  https://ftp.kegg.net/kegg/genes/organisms/lsal/*.pep.gz
Resolving ftp.kegg.net (ftp.kegg.net)... 133.18.96.71
Connecting to ftp.kegg.net (ftp.kegg.net)|133.18.96.71|:443... connected.
HTTP request sent, awaiting response... 401 Authorization Required
Authentication selected: Basic realm="KEGG HTTP Download Service"
Reusing existing connection to ftp.kegg.net:443.
HTTP request sent, awaiting response... 404 Not Found
2025-04-21 15:29:57 ERROR 404: Not Found.

--2025-04-21 15:29:58--  https://ftp.kegg.net/kegg/genes/organisms/dtx/*.pep.gz
Resolving ftp.kegg.net (ftp.kegg.net)... 133.18.96.71
Connecting to ftp.kegg.net (ftp.kegg.net)|133.18.96.71|:443... connected.


HTTP request sent, awaiting response... 401 Authorization Required
Authentication selected: Basic realm="KEGG HTTP Download Service"
Reusing existing connection to ftp.kegg.net:443.
HTTP request sent, awaiting response... 404 Not Found
2025-04-21 15:29:58 ERROR 404: Not Found.

--2025-04-21 15:29:58--  https://ftp.kegg.net/kegg/genes/organisms/hsz/*.pep.gz
Resolving ftp.kegg.net (ftp.kegg.net)... 133.18.96.71
Connecting to ftp.kegg.net (ftp.kegg.net)|133.18.96.71|:443... 

connected.
HTTP request sent, awaiting response... 401 Authorization Required
Authentication selected: Basic realm="KEGG HTTP Download Service"
Reusing existing connection to ftp.kegg.net:443.
HTTP request sent, awaiting response... 404 Not Found
2025-04-21 15:29:58 ERROR 404: Not Found.

--2025-04-21 15:29:58--  https://ftp.kegg.net/kegg/genes/organisms/hut/*.pep.gz
Resolving ftp.kegg.net (ftp.kegg.net)... 133.18.96.71
Connecting to ftp.kegg.net (ftp.kegg.net)|133.18.96.71|:443... connected.


HTTP request sent, awaiting response... 401 Authorization Required
Authentication selected: Basic realm="KEGG HTTP Download Service"
Reusing existing connection to ftp.kegg.net:443.
HTTP request sent, awaiting response... 404 Not Found
2025-04-21 15:29:59 ERROR 404: Not Found.

--2025-04-21 15:29:59--  https://ftp.kegg.net/kegg/genes/organisms/adl/*.pep.gz
Resolving ftp.kegg.net (ftp.kegg.net)... 133.18.96.71
Connecting to ftp.kegg.net (ftp.kegg.net)|133.18.96.71|:443... 

connected.
HTTP request sent, awaiting response... 401 Authorization Required
Authentication selected: Basic realm="KEGG HTTP Download Service"
Reusing existing connection to ftp.kegg.net:443.
HTTP request sent, awaiting response... 404 Not Found
2025-04-21 15:29:59 ERROR 404: Not Found.

--2025-04-21 15:29:59--  https://ftp.kegg.net/kegg/genes/organisms/saq/*.pep.gz
Resolving ftp.kegg.net (ftp.kegg.net)... 133.18.96.71
Connecting to ftp.kegg.net (ftp.kegg.net)|133.18.96.71|:443... connected.


HTTP request sent, awaiting response... 401 Authorization Required
Authentication selected: Basic realm="KEGG HTTP Download Service"
Reusing existing connection to ftp.kegg.net:443.
HTTP request sent, awaiting response... 404 Not Found
2025-04-21 15:30:00 ERROR 404: Not Found.

--2025-04-21 15:30:00--  https://ftp.kegg.net/kegg/genes/organisms/mdp/*.pep.gz
Resolving ftp.kegg.net (ftp.kegg.net)... 133.18.96.71
Connecting to ftp.kegg.net (ftp.kegg.net)|133.18.96.71|:443... 